# HYPER-3, 5 — The sequential design: boundaries, and what they cost

Everything so far has been preparation for one sentence in the protocol:

> *If a dose arm is making people markedly worse than the standard of care — overall or
> in any pre-specified age band — that arm stops.*

Turning that into a design means answering four questions with numbers, and
`axiom.design.sequential` answers all four from the same object:

1. **When do we look?** Every second calendar week from week 12 — once three eighths of
   the safety endpoints are in. Earlier than that, a stratum-level contrast rests on a
   handful of units per arm and has no normal approximation worth acting on.
2. **What counts as "markedly worse"?** A posterior probability against a stated margin,
   not a p-value — notebook 3 showed a stratum-level significance test cannot fire until
   half the trial has accrued.
3. **What does the rule cost when nothing is wrong?** `crossing_probabilities` computes
   it exactly, per contrast and for the family of twelve.
4. **What does it buy when something is?** `operating_characteristics`, converted into
   the only currency that matters here: unit-weeks of exposure to a harmful dose.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

import hyper3 as h
from axiom.core import AcceptanceRegion, clopper_pearson
from axiom.design import (
    Boundary, CrossingProbabilities, DecisionSpec, LookSchedule, OperatingCharacteristics,
    StoppingRule, alpha_spending, crossing_probabilities, difference_se, eig_gaussian,
    evoi_gaussian, evpi_gaussian, harm_boundary, information_fractions, monitor,
    obrien_fleming, operating_characteristics, pocock, preposterior_sd, spending,
)

from axiom.display import enable

enable();  # every axiom result renders itself from here on

SD_WINDOW = 6.0            # sd of a unit's four-week averaged endpoint (notebook 1 measured 6.02)
HARM_MARGIN = 2.0          # mmHg: "markedly worse" is worse than the control arm by this much
HARM_PROBABILITY = 0.95    # stop when the posterior probability of that reaches 95 %
EFFICACY_ALPHA = 0.025     # one-sided, split three ways across the doses
trial = h.trial(seed=20260821)

## 1. The look schedule comes from the accrual, not from the calendar

A unit joins the safety analysis when it completes the weeks 5–8 window. Enrollment is
uniform over 16 calendar weeks, so completed windows accrue in a straight line and the
planned information fractions are eighths of the total — the first review at three of
them, the last when every window is closed. `information_fractions` turns the planned
counts into that scale; the realized accrual is printed beside it.

In [ ]:
LOOK_WEEKS = list(range(12, 24, 2))
planned_counts = [h.N_UNITS * (w - 6) / 16 for w in LOOK_WEEKS]
INFORMATION = information_fractions(planned_counts, h.N_UNITS)
realized = h.accrual(trial, LOOK_WEEKS, endpoint="safety")
schedule = LookSchedule(labels=tuple(f"week_{w}" for w in LOOK_WEEKS), information=INFORMATION)
print(f"{'calendar week':>14} {'planned units':>14} {'planned t':>10} "
      f"{'realized units':>15} {'realized t':>11}")
for week, planned, t, (_, row) in zip(LOOK_WEEKS, planned_counts, INFORMATION,
                                      realized.iterrows(), strict=True):
    print(f"{week:14d} {planned:14.0f} {t:10.3f} {int(row['units']):15d} {row['share']:11.3f}")
print("\nincrements:", [round(d, 3) for d in schedule.increments])

## 2. The harm boundary is the protocol sentence, translated

`harm_boundary` takes the rule as written — *stop when the posterior probability that
this arm is worse than control by more than 2 mmHg reaches 95 %* — and returns the
threshold on `Z = effect / se` that implements it at each look. Because `se` shrinks as
information accrues, the threshold gets **stricter** over time: early on, a large
observed gap is cheap to come by, and the margin is doing the work of not being fooled
by it.

Twelve contrasts are monitored: each of three doses against the control arm overall,
and each of them inside each of three age bands. They differ only in their standard
error, so they differ only in how much the margin costs them.

In [ ]:
N_ARM = {arm: int(h.N_UNITS * k / h.BLOCK) for arm, k in h.ALLOCATION.items()}
N_STRATUM_ARM = {
    (stratum, arm): int(h.N_UNITS * h.STRATUM_SHARE[stratum] * k / h.BLOCK)
    for stratum in h.STRATA for arm, k in h.ALLOCATION.items()
}
contrasts: dict[tuple[str, str], float] = {}
for arm in h.ARMS[1:]:
    n_dose, n_control = N_ARM[arm], N_ARM["standard_of_care"]
    contrasts[("all", arm)] = difference_se(n_dose + n_control, sd=SD_WINDOW,
                                            allocation=n_dose / (n_dose + n_control))
    for stratum in h.STRATA:
        n_dose = N_STRATUM_ARM[(stratum, arm)]
        n_control = N_STRATUM_ARM[(stratum, "standard_of_care")]
        contrasts[(stratum, arm)] = difference_se(n_dose + n_control, sd=SD_WINDOW,
                                                  allocation=n_dose / (n_dose + n_control))

rules: dict[tuple[str, str], StoppingRule] = {}
for key, se in contrasts.items():
    boundary = harm_boundary(HARM_PROBABILITY, INFORMATION, margin=HARM_MARGIN,
                             se_at_full_information=se)
    rules[key] = StoppingRule(name=f"harm|{key[0]}|{key[1]}", looks=schedule,
                              boundaries=(boundary,))
example = rules[("age_51_plus", "dose_40")].boundaries[0]
print("40 mg inside the 51+ band — se at full information",
      round(contrasts[("age_51_plus", "dose_40")], 3), h.OUTCOME_UNIT)
print(f"{'look':>5} {'t':>7} {'se':>7} {'Z threshold':>12} {'effect at it':>14} {'nominal p':>10}")
for k, t in enumerate(INFORMATION):
    se_k = contrasts[("age_51_plus", "dose_40")] / float(np.sqrt(t))
    print(f"{k + 1:5d} {t:7.3f} {se_k:7.2f} {example.z[k]:12.3f} "
          f"{example.z[k] * se_k:14.2f} {example.nominal_alpha(k):10.4f}")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "On the Z scale — what the rule tests", "On the mmHg scale — what a committee reads"))
for stratum, arm, colour in (("all", "dose_40", "#5b6472"),
                             ("age_25_35", "dose_40", "#3aa17e"),
                             ("age_36_50", "dose_40", "#c9a227"),
                             ("age_51_plus", "dose_40", "#b5453b")):
    boundary = rules[(stratum, arm)].boundaries[0]
    label = "all units" if stratum == "all" else h.STRATUM_LABEL[stratum]
    fig.add_trace(go.Scatter(x=INFORMATION, y=boundary.z, mode="lines+markers", name=label,
                             line={"color": colour, "width": 2.4}), row=1, col=1)
    ses = contrasts[(stratum, arm)] / np.sqrt(np.asarray(INFORMATION))
    fig.add_trace(go.Scatter(x=INFORMATION, y=-np.asarray(boundary.z) * ses, mode="lines+markers",
                             showlegend=False, line={"color": colour, "width": 2.4}), row=1, col=2)
fig.add_hline(y=HARM_MARGIN, line={"color": "#111", "dash": "dot"}, row=1, col=2,
              annotation_text=f"the {HARM_MARGIN:g} mmHg margin")
fig.add_hline(y=h.intent_to_treat_contrast(40.0, "age_51_plus"),
              line={"color": "#d1483f", "dash": "dash"}, row=1, col=2,
              annotation_text="the real 40 mg harm in the 51+ band")
for col in (1, 2):
    fig.update_xaxes(title_text="information fraction", row=1, col=col, gridcolor=h.GRID)
fig.update_yaxes(title_text="Z threshold (negative is harm)", row=1, col=1, gridcolor=h.GRID)
fig.update_yaxes(title_text=f"observed harm needed to stop ({h.OUTCOME_UNIT})", row=1, col=2,
                 gridcolor=h.GRID)
fig.update_layout(height=430, template="plotly_white",
                  title="The same harm rule, on the two scales it lives on",
                  legend={"orientation": "h", "y": 1.12, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig

The right-hand panel is the one to argue about. At the first review the 51+ contrast
has to see 5.7 mmHg of harm before the rule fires; by the last it needs 4.3. The real
harm is 5.8 — just over the first threshold and comfortably over the rest — so this
design acts, and the interesting question is *when*, not whether.

## 3. What the rule spends when nothing is wrong

A posterior-probability rule states a posterior, not an error rate.
`crossing_probabilities` supplies the error rate.

In [ ]:
rows = []
for (stratum, arm), rule in rules.items():
    spent = crossing_probabilities(rule, 0.0)
    assert isinstance(spent, CrossingProbabilities)
    rows.append({"stratum": "all units" if stratum == "all" else h.STRATUM_LABEL[stratum],
                 "arm": h.ARM_LABEL[arm], "se": contrasts[(stratum, arm)],
                 "false_stop": spent.cumulative("harm"),
                 "first_look_share": spent.per_look["harm"][0] / spent.cumulative("harm")})
spending_table = pd.DataFrame(rows)
print("probability of a harm stop when the arm is identical to control")
print(spending_table.pivot(index="stratum", columns="arm", values="false_stop").round(4).to_string())
print("\nA wider standard error pays more for the margin: 2 mmHg is a larger share of the")
print("51+ band's se than of the pooled arm's, so the pooled arm-level monitor is the")
print("conservative one - and, per notebook 1, the one that will never see this harm.")

In [ ]:
fig = h.figure("Where the false-stop probability is spent, look by look",
               "information fraction", "probability of a first crossing here", height=390)
for stratum, colour in (("all", "#5b6472"), ("age_25_35", "#3aa17e"),
                        ("age_36_50", "#c9a227"), ("age_51_plus", "#b5453b")):
    per_look = crossing_probabilities(rules[(stratum, "dose_40")], 0.0).per_look["harm"]
    fig.add_trace(go.Bar(x=[f"{t:.3f}" for t in INFORMATION], y=per_look,
                         name="all units" if stratum == "all" else h.STRATUM_LABEL[stratum],
                         marker_color=colour))
fig.update_layout(barmode="group")
fig

### Against the rule nobody writes down

The alternative a monitoring committee reaches for without noticing is *test at 5 % at
every review and stop the first time it clears*. That rule is more sensitive — and it
has no error control, because it gets six chances. `crossing_probabilities` prices both
on the same scale.

In [ ]:
naive = Boundary(kind="harm", side="lower", z=(float(stats.norm.ppf(0.025)),) * schedule.n_looks)
key = ("age_51_plus", "dose_40")
se_key = contrasts[key]
true_harm = h.intent_to_treat_contrast(40.0, "age_51_plus")
print(f"{'rule':34s} {'null spend':>11} {'P(stop | the real harm)':>24} {'E[info]':>9}")
for label, rule in (("test at 5 % every review",
                     StoppingRule(name="naive", looks=schedule, boundaries=(naive,))),
                    (f"HYPER-3: {HARM_MARGIN:g} mmHg margin at {HARM_PROBABILITY:.0%}", rules[key])):
    null = crossing_probabilities(rule, 0.0).cumulative("harm")
    against = operating_characteristics(rule, -true_harm / se_key)
    print(f"{label:34s} {null:11.4f} {against.crossings.cumulative('harm'):24.3f} "
          f"{against.expected_information:9.3f}")
naive_spend = crossing_probabilities(
    StoppingRule(name="naive", looks=schedule, boundaries=(naive,)), 0.0).cumulative("harm")
print(f"\nacross twelve contrasts, if they were independent, at least one false stop:")
print(f"   test at 5 % every review : {1 - (1 - naive_spend) ** len(rules):.3f}")
print(f"   HYPER-3 rule             : "
      f"{1 - float(np.prod([1 - r for r in spending_table['false_stop']])):.3f}")
print("\nThe boundary gives up about a tenth of the sensitivity and spends a sixth of the")
print("error per contrast. Across the family that is the difference between a trial that")
print("trips something half the time for no reason and one that does not.")

## 4. Twelve contrasts, one trial

Each contrast is honest on its own. The family is not: with twelve of them, a trial in
which nothing is wrong will still trip *something* far more often than any single
number above suggests. The contrasts are correlated — the three doses inside a stratum
share a control arm, which correlates their Z statistics at 1/3 — so the independence
bound overstates it. Simulating the canonical joint distribution directly gives the
real number.

In [ ]:
def simulate_family(n_trials: int, drifts: dict[tuple[str, str], float], seed: int) -> pd.DataFrame:
    """Draw correlated Brownian paths for all twelve contrasts and apply their rules."""
    rng = np.random.default_rng(seed)
    keys = list(rules)
    increments = np.asarray(schedule.increments)
    # Within a stratum the three dose contrasts share the control arm: with a control
    # twice the size of each dose arm, corr(Z_i, Z_j) = (1/n_c) / (1/n_d + 1/n_c) = 1/3.
    groups: dict[str, list[int]] = {}
    for index, (stratum, _) in enumerate(keys):
        groups.setdefault(stratum, []).append(index)
    covariance = np.eye(len(keys))
    for members in groups.values():
        for i in members:
            for j in members:
                if i != j:
                    covariance[i, j] = 1.0 / 3.0
    chol = np.linalg.cholesky(covariance)
    mu = np.asarray([drifts.get(key, 0.0) for key in keys])

    stops = np.zeros((n_trials, len(keys)), dtype=bool)
    look_of = np.full((n_trials, len(keys)), schedule.n_looks, dtype=int)
    for trial_index in range(n_trials):
        shocks = rng.normal(size=(schedule.n_looks, len(keys))) @ chol.T
        steps = mu * increments[:, None] + shocks * np.sqrt(increments)[:, None]
        b = np.cumsum(steps, axis=0)
        z = b / np.sqrt(np.asarray(schedule.information))[:, None]
        for index, key in enumerate(keys):
            path = monitor(rules[key], [float(v) for v in z[:, index]])
            if path.stopped_at is not None:
                stops[trial_index, index] = True
                look_of[trial_index, index] = path.stopped_at
    return pd.DataFrame({"key": [f"{a}|{b}" for a, b in keys],
                         "stop_rate": stops.mean(axis=0),
                         "mean_look": look_of.mean(axis=0)}), stops


N_TRIALS = 4000
table, stops = simulate_family(N_TRIALS, {}, seed=11)
any_stop = float(stops.any(axis=1).mean())
independence_bound = 1.0 - float(np.prod([1.0 - r for r in spending_table["false_stop"]]))
print(f"{N_TRIALS} null trials, twelve contrasts monitored")
print(f"  P(at least one harm stop)          = {any_stop:.3f}")
print(f"  independence bound                 = {independence_bound:.3f}")
print(f"  largest single-contrast rate       = {spending_table['false_stop'].max():.3f}")

region = clopper_pearson(N_TRIALS, float(spending_table.loc[
    spending_table["stratum"] == "25-35"].query("arm == '40 mg'")["false_stop"].iloc[0]), 1e-3)
assert isinstance(region, AcceptanceRegion)
observed = int(stops[:, list(rules).index(("age_25_35", "dose_40"))].sum())
print(f"\ncheck: the 25-35 / 40 mg contrast crossed {observed} times; the exact acceptance")
print(f"region for its analytic rate is [{region.lower}, {region.upper}] -> "
      f"{'inside' if region.accepts(observed) else 'OUTSIDE'}")

About one null trial in twelve trips a harm boundary somewhere — well above any single
contrast's rate, and well below the independence bound, because the three doses in a
stratum share a control arm. **HYPER-3 does not correct for it**, and that is a decision
rather than an oversight: a false stop costs one arm
of a phase II, and a missed harm costs units their pressure control for months. The
asymmetry is stated in the protocol, the number above is stated with it, and the
committee reads a stopped arm as a signal to investigate rather than as a verdict.

## 5. What the rule buys

`operating_characteristics` at a grid of drifts, converted to the scale the protocol
was written on. The x-axis is the *true* harm in mmHg; the drift is that divided by the
contrast's standard error at full information.

In [ ]:
harms = np.linspace(0.0, 10.0, 41)
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Probability the arm is stopped", "Information used before stopping"))
for stratum, colour in (("all", "#5b6472"), ("age_25_35", "#3aa17e"),
                        ("age_36_50", "#c9a227"), ("age_51_plus", "#b5453b")):
    rule = rules[(stratum, "dose_40")]
    se = contrasts[(stratum, "dose_40")]
    ocs = [operating_characteristics(rule, -float(harm) / se) for harm in harms]
    label = "all units" if stratum == "all" else h.STRATUM_LABEL[stratum]
    fig.add_trace(go.Scatter(x=harms, y=[oc.crossings.cumulative("harm") for oc in ocs],
                             mode="lines", name=label, line={"color": colour, "width": 2.6}),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=harms, y=[oc.expected_information for oc in ocs], mode="lines",
                             showlegend=False, line={"color": colour, "width": 2.6}), row=1, col=2)
for col in (1, 2):
    fig.add_vline(x=h.intent_to_treat_contrast(40.0, "age_51_plus"),
                  line={"color": "#111", "dash": "dash"}, row=1, col=col)
    fig.update_xaxes(title_text=f"true harm ({h.OUTCOME_UNIT} worse than control)", row=1, col=col,
                     gridcolor=h.GRID)
fig.update_yaxes(title_text="P(stop for harm)", row=1, col=1, gridcolor=h.GRID)
fig.update_yaxes(title_text="expected information fraction", row=1, col=2, gridcolor=h.GRID)
fig.update_layout(height=420, template="plotly_white",
                  title="Operating characteristics of the harm rule",
                  legend={"orientation": "h", "y": 1.12, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig

In [ ]:
oldest = rules[("age_51_plus", "dose_40")]
true_harm = h.intent_to_treat_contrast(40.0, "age_51_plus")
oc = operating_characteristics(oldest, -true_harm / contrasts[("age_51_plus", "dose_40")])
assert isinstance(oc, OperatingCharacteristics)
print(f"against the real 40 mg harm of {true_harm:.2f} {h.OUTCOME_UNIT} in the 51+ band:")
print(f"  P(stop)                = {oc.crossings.cumulative('harm'):.3f}")
print(f"  expected information   = {oc.expected_information:.3f}")
print(f"  expected looks         = {oc.expected_looks:.2f} of {schedule.n_looks}")
print("  stop probability by look:",
      [round(p, 3) for p in oc.crossings.by_look()])

## 6. The currency that matters: unit-weeks on a harmful dose

Expected information is an abstraction. Turn it into the thing the protocol sentence is
about — how many unit-weeks of exposure to a harmful dose the trial produces — using
the enrollment schedule directly. Stopping at calendar week `w` ends dosing for the
units already randomized to that cell and re-randomizes everyone after it.

In [ ]:
cell = trial.units[(trial.units["stratum"] == "age_51_plus") & (trial.units["arm"] == "dose_40")]
entry = cell["enrolled_week"].to_numpy()


def unit_weeks(stop_week: float) -> float:
    """Unit-weeks of 40 mg taken in the 51+ band if the arm stops at ``stop_week``."""
    on_drug = np.clip(np.minimum(stop_week, entry + h.FOLLOW_UP_WEEKS) - entry, 0.0, None)
    return float(on_drug.sum())


fixed_sample = unit_weeks(h.TRIAL_WEEKS)
by_look = oc.crossings.by_look()
expected = sum(p * unit_weeks(w) for p, w in zip(by_look, LOOK_WEEKS, strict=True)) \
    + oc.crossings.continue_probability * fixed_sample
print(f"fixed-sample trial: {fixed_sample:,.0f} unit-weeks on 40 mg in the 51+ band")
print(f"sequential design : {expected:,.0f} expected unit-weeks — "
      f"{1 - expected / fixed_sample:.0%} less exposure")
print(f"units never randomized to the arm at all: about "
      f"{sum(p * (entry > w).sum() for p, w in zip(by_look, LOOK_WEEKS, strict=True)):.0f} of {len(cell)}")

In [ ]:
fig = h.figure("Exposure to a harmful arm: what the boundary is actually buying", "",
               "unit-weeks on 40 mg in the 51+ band", height=400)
margins = (0.0, 1.0, 2.0, 3.0, 4.0, 5.0)
sequential, false_stops = [], []
for margin in margins:
    boundary = harm_boundary(HARM_PROBABILITY, INFORMATION, margin=margin,
                             se_at_full_information=contrasts[("age_51_plus", "dose_40")])
    rule = StoppingRule(name=f"margin_{margin:g}", looks=schedule, boundaries=(boundary,))
    result = operating_characteristics(rule, -true_harm / contrasts[("age_51_plus", "dose_40")])
    sequential.append(sum(p * unit_weeks(w)
                          for p, w in zip(result.crossings.by_look(), LOOK_WEEKS, strict=True))
                      + result.crossings.continue_probability * fixed_sample)
    false_stops.append(crossing_probabilities(rule, 0.0).cumulative("harm"))
fig.add_trace(go.Bar(x=[f"{m:g} mmHg" for m in margins], y=sequential, name="expected exposure",
                     marker_color="#b5453b"))
fig.add_hline(y=fixed_sample, line={"color": "#111", "dash": "dash"},
              annotation_text=f"fixed sample: {fixed_sample:,.0f}")
fig.add_trace(go.Scatter(x=[f"{m:g} mmHg" for m in margins], y=false_stops,
                         mode="lines+markers", name="false-stop rate per contrast",
                         yaxis="y2", line={"color": "#2f7fd1", "width": 2.6}))
fig.update_layout(yaxis2={"overlaying": "y", "side": "right", "showgrid": False,
                          "title": "P(stop | arm identical to control)", "rangemode": "tozero"},
                  xaxis_title="harm margin in the rule")
for margin, exposure, false_stop in zip(margins, sequential, false_stops, strict=True):
    print(f"margin {margin:>4.1f} {h.OUTCOME_UNIT}: expected exposure {exposure:7,.0f} unit-weeks "
          f"({1 - exposure / fixed_sample:5.0%} saved), false-stop rate {false_stop:.4f}")
fig

The margin is the whole trade, and it is priced: a rule with no margin stops the harmful
arm almost immediately and stops one null arm in six; a 4 mmHg margin almost never fires
falsely and leaves most of the exposure in place. HYPER-3 takes 2 mmHg because that is
the smallest increase in systolic pressure the protocol is willing to call clinically
meaningful, and the numbers above are what that choice costs and buys.

## 7. The other rule: efficacy and futility on the primary endpoint

Safety monitoring is one-sided and generous. The *efficacy* rule is the opposite: a
one-sided 0.025 spent across three doses, on a Lan–DeMets O'Brien–Fleming shape so that
an early stop demands an extreme result and the final critical value stays close to the
fixed-sample one.

In [ ]:
EFFICACY_LOOKS = (0.5, 0.75, 1.0)
efficacy_schedule = LookSchedule(labels=("week_18", "week_22", "week_26"),
                                 information=EFFICACY_LOOKS)
per_dose_alpha = EFFICACY_ALPHA / 3
shapes = {
    "Lan-DeMets OBF": alpha_spending(per_dose_alpha, EFFICACY_LOOKS, family="obrien_fleming",
                                     side="upper"),
    "Lan-DeMets Pocock": alpha_spending(per_dose_alpha, EFFICACY_LOOKS, family="pocock", side="upper"),
    "O'Brien-Fleming": obrien_fleming(per_dose_alpha, EFFICACY_LOOKS, side="upper"),
    "Pocock": pocock(per_dose_alpha, EFFICACY_LOOKS, side="upper"),
}
se_primary = difference_se(N_ARM["dose_20"] + N_ARM["standard_of_care"], sd=SD_WINDOW,
                           allocation=N_ARM["dose_20"] / (N_ARM["dose_20"] + N_ARM["standard_of_care"]))
drift_at_4 = 4.0 / se_primary
print(f"one-sided alpha per dose: {per_dose_alpha:.4f}; se at full information "
      f"{se_primary:.3f}; drift at a 4 {h.OUTCOME_UNIT} benefit: {drift_at_4:.2f}")
print(f"\n{'shape':20} {'thresholds':>26} {'type I':>8} {'power':>7} {'E[info]':>9}")
for name, boundary in shapes.items():
    rule = StoppingRule(name=name, looks=efficacy_schedule, boundaries=(boundary,))
    null = crossing_probabilities(rule, 0.0).cumulative("efficacy")
    alt = operating_characteristics(rule, drift_at_4)
    thresholds = " ".join(f"{z:6.3f}" for z in boundary.z)
    print(f"{name:20} {thresholds:>26} {null:8.4f} "
          f"{alt.crossings.cumulative('efficacy'):7.3f} {alt.expected_information:9.3f}")

In [ ]:
fig = h.figure("Four ways to spend the same one-sided alpha", "information fraction",
               "Z threshold", height=390)
for (name, boundary), colour in zip(shapes.items(), ("#2f7fd1", "#8a63c4", "#3aa17e", "#c9a227")):
    fig.add_trace(go.Scatter(x=EFFICACY_LOOKS, y=boundary.z, mode="lines+markers", name=name,
                             line={"color": colour, "width": 2.4}))
fig.add_hline(y=float(stats.norm.isf(per_dose_alpha)), line={"color": "#111", "dash": "dot"},
              annotation_text="fixed-sample critical value")
fig

`spending` is the function the thresholds are solved against, and it is worth plotting
on its own: it is the whole difference between a design that can stop early cheaply and
one that cannot.

In [ ]:
grid = np.linspace(0.01, 1.0, 100)
fig = h.figure("Alpha-spending functions", "information fraction",
               "cumulative one-sided alpha spent", height=360)
for kind, colour in (("obrien_fleming", "#2f7fd1"), ("pocock", "#8a63c4"), ("power", "#c9a227")):
    fig.add_trace(go.Scatter(x=grid, y=[spending(kind, float(t), per_dose_alpha) for t in grid],
                             mode="lines", name=kind, line={"color": colour, "width": 2.6}))
fig.add_trace(go.Scatter(x=grid, y=[spending("power", float(t), per_dose_alpha, rho=3.0) for t in grid],
                         mode="lines", name="power, rho=3", line={"color": "#b5453b", "width": 2.0,
                                                                  "dash": "dash"}))
for t in EFFICACY_LOOKS:
    fig.add_vline(x=t, line={"color": "rgba(0,0,0,0.2)", "dash": "dot"})
fig

## 8. The whole rule, assembled

A `StoppingRule` composes the boundaries that are evaluated together. The 20 mg arm's
primary rule carries three: efficacy above, harm below, and a non-binding futility
boundary that lets the committee abandon an arm that is going nowhere. Non-binding means
the error-rate arithmetic pretends it is not there — the conservative convention — and
`binding_only=False` says what the rule as run actually does.

In [ ]:
harm_on_primary = harm_boundary(HARM_PROBABILITY, EFFICACY_LOOKS, margin=HARM_MARGIN,
                                se_at_full_information=se_primary)
futility = Boundary(kind="futility", side="lower", z=(-0.5, 0.3, 0.9), binding=False)
combined = StoppingRule(name="hyper3_dose_20_primary", looks=efficacy_schedule,
                        boundaries=(shapes["Lan-DeMets OBF"], harm_on_primary, futility))
for look, label in enumerate(efficacy_schedule.labels):
    low, high = combined.continuation(look)
    print(f"{label}: continue while {low:+.3f} < Z < {high:+.3f}  "
          f"(t = {EFFICACY_LOOKS[look]:.2f})")
for drift, label in ((0.0, "null"), (drift_at_4, "a 4 mmHg benefit"),
                     (-3.0 / se_primary, "a 3 mmHg harm")):  # noqa: E501
    binding = crossing_probabilities(combined, drift)
    as_run = crossing_probabilities(combined, drift, binding_only=False)
    print(f"\n{label} (drift {drift:+.2f}):")
    print(f"  efficacy {binding.cumulative('efficacy'):.4f} | harm {binding.cumulative('harm'):.4f}"
          f" | continue {binding.continue_probability:.4f}   [futility ignored]")
    print(f"  efficacy {as_run.cumulative('efficacy'):.4f} | harm {as_run.cumulative('harm'):.4f}"
          f" | futility {as_run.cumulative('futility'):.4f}   [as the committee runs it]")

## 9. What the interim information is worth

The decision the interim serves is whether to carry 40 mg into a phase III. If the
oldest band is being harmed and the trial does not find out, the phase III is run on the
wrong dose. `evoi_gaussian` prices the interim against that decision: EVPI is what a
clairvoyant would be worth, EVSI what *this* look is worth.

In [ ]:
PRIOR_MEAN, PRIOR_SD = 3.0, 4.0      # mmHg of reduction expected at 40 mg in the 51+ band
PHASE_III_COST = 80_000_000.0        # what running the wrong dose through phase III costs
decision = DecisionSpec(name="carry_40mg_to_phase_three", threshold=0.0,
                        value_per_outcome_unit=PHASE_III_COST / 4.0, numeraire="USD")
se_full = contrasts[("age_51_plus", "dose_40")]
print(f"{'look':>5} {'t':>7} {'se':>7} {'EIG (nats)':>11} {'EVSI':>16} {'preposterior sd':>17}")
for k, t in enumerate(INFORMATION):
    se_k = se_full / float(np.sqrt(t))
    value = evoi_gaussian(decision, PRIOR_MEAN, PRIOR_SD, se_k)
    print(f"{k + 1:5d} {t:7.3f} {se_k:7.2f} {eig_gaussian(PRIOR_SD, se_k):11.3f} "
          f"{value.evsi:16,.0f} {preposterior_sd(PRIOR_SD, se_k):17.3f}")
print(f"\nEVPI (a clairvoyant): {evpi_gaussian(decision, PRIOR_MEAN, PRIOR_SD):,.0f} USD")

In [ ]:
fig = h.figure("What each interim look is worth against the phase III decision",
               "information fraction", "expected value of the sample information (USD)", height=380)
ses = se_full / np.sqrt(np.asarray(INFORMATION))
fig.add_trace(go.Scatter(x=INFORMATION, y=[evoi_gaussian(decision, PRIOR_MEAN, PRIOR_SD, s).evsi
                                           for s in ses],
                         mode="lines+markers", name="EVSI", line={"color": "#2f7fd1", "width": 2.6}))
fig.add_hline(y=evpi_gaussian(decision, PRIOR_MEAN, PRIOR_SD), line={"color": "#111", "dash": "dash"},
              annotation_text="EVPI")
fig

## 10. When the looks do not happen when they were planned

Enrollment slips. A boundary built from a *shape* is invalidated by that; a boundary
built from a *spending function* is not — `alpha_spending` re-solves the remaining
thresholds against the information that actually accrued, and the total alpha is
unchanged.

In [ ]:
slipped = (0.42, 0.71, 1.0)
for label, fractions in (("as planned", EFFICACY_LOOKS), ("as it happened", slipped)):
    boundary = alpha_spending(per_dose_alpha, fractions, family="obrien_fleming", side="upper")
    print(f"{label:16s} t = {[round(t, 2) for t in fractions]}")
    print(f"{'':16s} z = {[round(z, 3) for z in boundary.z]}  "
          f"total spent {boundary.spent[-1]:.5f}")

## What this notebook decided

- The harm rule is the protocol sentence with a number in it: *stop when the posterior
  probability that this arm is worse than control by more than 2 mmHg reaches 95 %*.
  `harm_boundary` turns that into per-look thresholds; because the standard error
  shrinks, the rule tightens over time rather than loosening.
- Per contrast it stops a null arm between 0.1 % and 1.5 % of the time, depending on how
  wide that contrast's standard error is. Across twelve monitored contrasts, about one
  null trial in twelve trips something. HYPER-3 does not correct for it, deliberately,
  and states the number instead of hiding it.
- The rule nobody writes down — *test at 5 % on every review* — is about a tenth more
  sensitive and spends six times the error per contrast, which across the family is the
  difference between tripping something in half of all null trials and in one in twelve.
- Against the real 5.8 mmHg harm in the oldest band the rule fires with probability
  0.91, and expects to do so at about 54 % of the information — more than half of that
  probability at the very first review. Expected exposure to the harmful dose falls from
  672 unit-weeks to 213.
- The margin is the trade and it is priced: no margin saves 80 % of the exposure and
  trips one null contrast in eight; a 4 mmHg margin trips one in twenty-five hundred and
  saves 35 %. 2 mmHg is the clinical judgement, and the table is what it costs.
- The efficacy rule is a Lan–DeMets O'Brien–Fleming boundary on a Bonferroni-split
  one-sided alpha, chosen because it keeps almost all the fixed-sample power. Because it
  is a spending function and not a shape, a look that slips does not invalidate it.
- Notebook 6 runs this rule against the trial that actually happened.